# Lekcja 5 — 5G NR PUSCH (komponenty Sionna)

## Cel nauki
Nie budujemy PUSCH od zera — używamy gotowych bloków Sionna:
- **LDPC 5G** encoder/decoder
- **ResourceGrid** z parametrami zbliżonymi do NR
- **OFDMChannel** z modelem TDL/CDL

## PUSCH w skrócie
Physical Uplink Shared Channel — kanał danych użytkownika w górę (UE → gNB).

## Pipeline docelowy pracy magisterskiej
```
Bits → LDPC → QAM → Grid → OFDM → 3GPP Channel → Receiver → LLR → LDPC decode
```

## Co zostaje klasyczne po wprowadzeniu CNN?
**LDPC decoder** — sieć produkuje LLR, dekoder zostaje standardowy (modularność!).


In [ ]:
import sys
from pathlib import Path

# Dodaj src/ do PYTHONPATH
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import torch

try:
    import sionna as sn
    import sionna.phy
except ImportError as e:
    raise ImportError(
        "Brak Sionny. Uruchom z katalogu magisterka/: ./scripts/drun sync"
    ) from e

from src.utils.setup import print_environment, get_device

sn.phy.config.seed = 42
device = get_device()
print_environment()


## LDPC 5G — encoder i decoder

In [ ]:
k = 512   # informacyjne bity
n = 1024  # długość kodoword
coderate = k / n

encoder = sn.phy.fec.ldpc.LDPC5GEncoder(k, n)
decoder = sn.phy.fec.ldpc.LDPC5GDecoder(encoder, hard_out=True)

bits = sn.phy.mapping.BinarySource()([16, k])
codewords = encoder(bits)
print("bits:", bits.shape, "codewords:", codewords.shape)


## Transmisja zakodowana przez AWGN

In [ ]:
NUM_BPS = 2
constellation = sn.phy.mapping.Constellation("qam", NUM_BPS)
mapper = sn.phy.mapping.Mapper(constellation=constellation)
demapper = sn.phy.mapping.Demapper("app", constellation=constellation)
awgn = sn.phy.channel.AWGN()

x = mapper(codewords)
ebno_db = 3.0
no = sn.phy.utils.ebnodb2no(ebno_db, NUM_BPS, coderate)
y = awgn(x, no)
llr = demapper(y, no)
bits_hat = decoder(llr)

from src.utils.metrics import ber, bler
print(f"BER @ {ebno_db} dB:", ber(codewords, bits_hat))
print(f"BLER @ {ebno_db} dB:", bler(bits, bits_hat))


## Uwaga
Pełny PUSCH z OFDM + TDL + klasycznym receiverem budujemy w **06_classical_receiver.ipynb**.

## Ćwiczenie
Porównaj BLER z i bez LDPC przy Eb/N0 = 0…6 dB.

**Następna lekcja:** `06_classical_receiver.ipynb`
